# Freemarket Medallion Pipeline — orchestrator

This notebook **orchestrates and narrates**; all transformation logic lives in the tested
`src/` package (see `plan/SPEC.md` § Engineering Standards). Run top-to-bottom
(Restart & Run All) from an empty warehouse.

Layers built so far: **Bronze foundation** (six schemas) → **Silver `core`** (companies + groups unpick).

In [1]:
# Make the repo root importable when running from notebook/
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, warehouse, silver_core
from src.pipeline import build_foundation
config.WAREHOUSE_PATH

PosixPath('/Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb')

## 1. Foundation — six schemas
Connect (creates the file if absent) and create the schemas. Idempotent.

In [2]:
con = warehouse.connect()
build_foundation(con)

19:14:49 | INFO  | [foundation] schemas ready: raw, live, core, shape, data_mart, curated


('raw', 'live', 'core', 'shape', 'data_mart', 'curated')

## 2. Silver `core` — companies unpick
Flatten `companies.json` into `core.companies` (one row per `dcId`), exposing nested
registration / classification / financials / footprint fields and the `parent_group_id`
bridge key.

In [3]:
report = silver_core.build_companies(con)
print(report)

19:14:49 | INFO  | [silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


[silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


In [4]:
con.sql("""
    SELECT dc_id, legal_name, vertical, financials_currency,
           parent_group_id, parent_group_role
    FROM core.companies
    ORDER BY dc_id
    LIMIT 10
""").df()

,dc_id,legal_name,vertical,financials_currency,parent_group_id,parent_group_role
0,234850008297,Payment One Holding S.à r.l.,Holding,USD,167898894567,HOLDING
1,234854669542,Remitance Central Holdings Ltd,Holding,USD,167986897142,HOLDING
2,234939159769,Payment One Operating Company Ltd,MSB,USD,167898894567,OPERATING
3,234939159775,Remitance Central (Europe) Ltd,MSB,USD,167986897142,OPERATING
4,258599414982,Get Tranzzact (Europe) Ltd,MSB,USD,187752508616,OPERATING
5,258740050135,ConstructGame Technology Pvt Ltd,Technology,USD,187754310858,TECHNOLOGY
6,258740050139,Sanno Payments Operations Ltd,MSB,USD,187757905092,OPERATING
7,258863167682,ConstructGame Group Holdings Ltd,Holding,USD,187754310858,HOLDING
8,258929414331,ConstructGame Media Ltd,Marketing,USD,187754310858,MARKETING
9,258962745548,ConstructGame Group Holdings Ltd,Holding,USD,187754310858,HOLDING


## 3. Silver `core` — groups unpick
Flatten `groups.json` (records under `result.groups`) into `core.groups` (one row per
`groupId`), exposing profile / segmentation / lifecycle fields. The heterogeneous
`attributes` array (scalar **or** object `value`) is carried forward as JSON for the
Silver `shape` cleanup step.

In [5]:
report = silver_core.build_groups(con)
print(report)

19:14:50 | INFO  | [silver.core.groups] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


[silver.core.groups] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


In [6]:
con.sql("""
    SELECT group_id, display_name, commercial_tier, pod, vertical, active
    FROM core.groups
    ORDER BY group_id
""").df()

,group_id,display_name,commercial_tier,pod,vertical,active
0,167898894567,Payment One,Gold,MSB/PSP,MSB,True
1,167986897142,Remitance Central,Bronze,MSB/PSP,MSB,True
2,187752508616,Get Tranzzact,Gold,MSB/PSP,MSB,False
3,187754310858,ConstructGame,Gold,Gaming/iGaming/Gambling,Gaming,True
4,187757905092,Sanno Payments,Silver,MSB/PSP,MSB,True
5,192212183238,Tech Key,Bronze,MSB/PSP,MSB,True
6,192230405343,Koba2Maya,Silver,Gaming/iGaming/Gambling,Gaming,True
7,192315275482,Oni Group,Bronze,MSB/PSP,MSB,True
8,192400133342,NXT Services,Silver,Crypto/CFD/Forex,Crypto,True
9,192402023626,Cube Empire,Bronze,MSB/PSP,MSB,False


## Inspect the built warehouse
Reopen `submission/warehouse.duckdb` **read-only** to see the schemas and the tables built
so far — this is how you'd open the warehouse file from anywhere with the `duckdb` package.

In [7]:
con.close()
import duckdb
inspect = duckdb.connect(str(config.WAREHOUSE_PATH), read_only=True)
tables = inspect.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    ORDER BY table_schema, table_name
""").df()
inspect.close()
tables

,table_schema,table_name
0,core,companies
1,core,groups
